# MCP Transports: stdio, HTTP, Streamable

**Phase 03** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-03/09-mcp-transports.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q mcp uvicorn

import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    pass  # Running locally — set OPENAI_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("OPENAI_API_KEY")))

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
L09: MCP Transports - stdio, HTTP SSE, and Streamable HTTP

Demonstrates how to configure the same MCP server for all three transports.
Each transport section is self-contained. Run with different entry points
depending on which transport you want.

Usage:
    # stdio (Claude Desktop forks this process):
    python main.py

    # HTTP SSE:
    python main.py --transport sse

    # Streamable HTTP:
    python main.py --transport streamable
"""

import argparse
import sys
from mcp.server import FastMCP

### Shared server + tool definitions

In [ ]:
mcp = FastMCP("product-lookup")

PRODUCTS = {
    "p001": {"name": "Widget A", "price": 9.99, "stock": 142},
    "p002": {"name": "Widget B", "price": 24.99, "stock": 8},
    "p003": {"name": "Gadget X", "price": 149.00, "stock": 0},
}


@mcp.tool()
def get_product(product_id: str) -> dict:
    """Look up a product by ID."""
    if product_id not in PRODUCTS:
        return {"error": f"Product {product_id} not found"}
    return PRODUCTS[product_id]


@mcp.tool()
def list_products() -> list[dict]:
    """List all available products with IDs, names, prices, and stock."""
    return [{"id": k, **v} for k, v in PRODUCTS.items()]


# ---------------------------------------------------------------------------
# Transport 1: stdio
# Claude Desktop forks this process and communicates via stdin/stdout.
# IMPORTANT: never print anything to stdout except MCP protocol messages.
# Use stderr for debug output: print("debug", file=sys.stderr)
# ---------------------------------------------------------------------------

def run_stdio():
    print("Starting MCP server with stdio transport", file=sys.stderr)
    mcp.run(transport="stdio")


# ---------------------------------------------------------------------------
# Transport 2: HTTP SSE
# Server runs as an HTTP process. Client connects to /sse endpoint.
# Supports remote deployment and multiple sequential clients.
# Not suitable for concurrent sessions or bidirectional streaming.
# ---------------------------------------------------------------------------

def run_sse(host: str = "0.0.0.0", port: int = 8080):
    print(f"Starting MCP server with SSE transport on {host}:{port}", file=sys.stderr)
    mcp.run(transport="sse", host=host, port=port)


# ---------------------------------------------------------------------------
# Transport 3: Streamable HTTP
# The 2025 production standard. Adds session management, reconnect support,
# and bidirectional streaming. Use this for any multi-client or production
# deployment.
# ---------------------------------------------------------------------------

def run_streamable(host: str = "0.0.0.0", port: int = 8080):
    """
    Streamable HTTP requires an ASGI server (uvicorn) and a session manager.
    The session manager assigns each client a session ID and handles reconnects.
    """
    try:
        import uvicorn
        from starlette.applications import Starlette
        from starlette.routing import Mount
        from mcp.server.streamable_http import StreamableHTTPSessionManager
    except ImportError:
        print(
            "ERROR: Streamable HTTP requires uvicorn and starlette.\n"
            "Install: pip install uvicorn starlette",
            file=sys.stderr,
        )
        sys.exit(1)

    # event_store=None uses an in-memory store.
    # Swap for a Redis-backed store in production for cross-process sessions.
    session_manager = StreamableHTTPSessionManager(
        app=mcp,
        event_store=None,
    )

    app = Starlette(
        routes=[
            Mount("/mcp", app=session_manager.asgi_app()),
        ]
    )

    print(
        f"Starting MCP server with Streamable HTTP transport on {host}:{port}/mcp",
        file=sys.stderr,
    )
    uvicorn.run(app, host=host, port=port)

### Transport decision helper (the USE IT section)

In [ ]:
from dataclasses import dataclass


@dataclass
class DeploymentContext:
    remote: bool           # Server is on a different machine than the client
    multi_client: bool     # More than one client connects concurrently
    needs_streaming: bool  # Server pushes partial results or progress updates
    production: bool       # Deployed beyond local dev


def choose_transport(ctx: DeploymentContext) -> tuple[str, str]:
    """
    Returns (transport_name, reason).
    Use this at architecture time, not at runtime.
    """
    if not ctx.remote:
        return "stdio", (
            "Same-machine deployment. Client forks the server process via stdin/stdout."
        )

    if ctx.multi_client or ctx.production or ctx.needs_streaming:
        return "streamable_http", (
            "Remote + production or multi-client: Streamable HTTP provides "
            "session management, reconnect support, and bidirectional streaming."
        )

    return "sse", (
        "Remote but simple: HTTP SSE is sufficient for a single client or "
        "low-traffic remote deployment without streaming requirements."
    )

### Demo: show the decision function output for three typical contexts

In [ ]:
def demo_decision_function():
    contexts = [
        (
            "Local Claude Desktop plugin",
            DeploymentContext(
                remote=False, multi_client=False, needs_streaming=False, production=False
            ),
        ),
        (
            "Simple remote deployment, one team",
            DeploymentContext(
                remote=True, multi_client=False, needs_streaming=False, production=False
            ),
        ),
        (
            "Hosted MCP server, multiple teams, long-running tools",
            DeploymentContext(
                remote=True, multi_client=True, needs_streaming=True, production=True
            ),
        ),
    ]

    print("\n=== Transport Decision Examples ===\n")
    for label, ctx in contexts:
        transport, reason = choose_transport(ctx)
        print(f"Context: {label}")
        print(f"  -> {transport}")
        print(f"     {reason}")
        print()


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

### Demo

In [ ]:
parser = argparse.ArgumentParser(description="MCP transport demo")
parser.add_argument(
    "--transport",
    choices=["stdio", "sse", "streamable", "demo"],
    default="demo",
    help="Which transport to run, or 'demo' to show the decision function",
)
parser.add_argument("--host", default="0.0.0.0")
parser.add_argument("--port", type=int, default=8080)
args = parser.parse_args()

if args.transport == "stdio":
    run_stdio()
elif args.transport == "sse":
    run_sse(args.host, args.port)
elif args.transport == "streamable":
    run_streamable(args.host, args.port)
else:
    demo_decision_function()

---

## Self-Check

Answer these without running code first:

**1. Why does moving the server to a remote VM break the stdio transport?**

- A. The MCP protocol is not supported over TCP/IP networks.
- B. stdio requires the client to fork the server as a local subprocess; it cannot connect to a process on another machine.
- C. Claude Desktop does not support MCP servers running on Linux VMs.
- D. The server needs to be recompiled with network support enabled.

**2. Which transport is the simplest correct choice for this scenario?**

- A. stdio, because it is the most reliable.
- B. Streamable HTTP, because it is always the safest choice.
- C. HTTP SSE, because the server is remote and requirements are simple.
- D. No transport works for single-user remote deployments.

**3. What does switching to Streamable HTTP fix in this situation?**

- A. It makes the data export tool run faster by using parallel processing.
- B. It allows the server to push progress updates during the 45-60 second window, preventing inactivity timeouts.
- C. It automatically retries the tool call if it times out.
- D. It increases the maximum allowed tool execution time to 5 minutes.

**4. What went wrong and how do you fix it?**

- A. The print statements crash the server. Use logging instead.
- B. Print to stdout corrupts the MCP protocol stream. Route debug output to stderr: print(..., file=sys.stderr).
- C. Claude Desktop blocks servers that produce extra output for security reasons.
- D. The print statements cause a race condition in the stdio pipe.

**5. Why does a rolling deploy break existing sessions and what is the fix?**

- A. Streamable HTTP does not support rolling deploys. Use a blue/green deploy strategy.
- B. In-memory session state is lost when the process restarts. Use a Redis-backed event store so sessions survive across process restarts.
- C. The clients need to reconnect with a new session ID after any server restart.
- D. Session persistence requires the client to store the session on disk.

**6. What nginx configuration change fixes this?**

- A. Increase the worker_processes count so more connections can be handled.
- B. Enable the ssl_session_cache directive to keep connections warm.
- C. Set proxy_read_timeout to a value larger than the longest expected tool execution time.
- D. Switch from nginx to Apache, which handles SSE connections better.

**7. How should they structure the server to support both transports from one codebase?**

- A. This is not possible. Each transport requires separate server code.
- B. Register all tools on a shared FastMCP instance, then select the transport at startup via an argument or environment variable.
- C. Use stdio for Claude Desktop and rewrite the tools as a REST API for CI/CD.
- D. Wrap the stdio server in a subprocess manager that exposes HTTP on top.

_Answers are in `checks.json` in the lesson directory._